# Persian News Headlines: Sentiment, Topic, and Agreement

**Beginner Colab project | Persian headlines | Human annotation + LLM classification**

In this project you will:
1. Upload the clean 2,000-headline Excel file and your class-annotated 100-row gold file.
2. Measure agreement between two human annotators with percent agreement and Cohen’s kappa.
3. Write and revise a short classification prompt in Persian.
4. Ask a Gemini model to assign one sentiment and one main topic to each headline.
5. Compare the model with the human consensus labels, then annotate all 2,000 headlines.
6. Make simple charts and download the annotated Excel file.

**Important:** The 100-row file is a blank annotation template. Two people should label it independently; discuss disagreements and fill the consensus columns before uploading it here. The API step requires a Gemini API key. API use may have quota or billing limits; check your Google AI Studio account before running the full sample.

## The two tasks

**Sentiment (3 labels):** Is the event or outcome framed positively, negatively, or neutrally in the headline? Judge the wording in the headline; do not infer an outlet’s political position. Use neutral when the headline is mainly factual or has no clear positive/negative evaluation.

**Topic (one main topic):** Choose the single best-fitting subject for the headline. Use `سایر / نامشخص` only when no category fits or the headline gives too little information.

The starting topics are suggestions. Choose and define the final categories **before** human annotation. Keep labels mutually understandable and apply the same codebook throughout. Each row gets one label for each task.

In [ ]:
# Install the small set of libraries used in this notebook.
%pip -q install -U google-genai openpyxl arabic-reshaper python-bidi seaborn

In [ ]:
import io, json, time, math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import arabic_reshaper
from bidi.algorithm import get_display
from IPython.display import display, Markdown
from google.colab import files, userdata
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from sklearn.metrics import cohen_kappa_score, confusion_matrix

pd.set_option('display.max_colwidth', 140)
sns.set_theme(style='whitegrid', palette='deep')
print('Ready. Next, upload the two Excel workbooks.')

## 1. Upload the two Excel workbooks

Upload `headlines_2000_sample.xlsx` first. Then upload your completed `gold_annotation_template_100.xlsx`. In the gold workbook, keep the sheet names and column names unchanged. Fill the two rater columns independently, then fill the consensus columns after discussion.

In [ ]:
print('Upload the 2,000-headline workbook')
data_upload = files.upload()
if not data_upload:
    raise ValueError('No file uploaded. Please upload headlines_2000_sample.xlsx.')
data_name = next(iter(data_upload))
dataset = pd.read_excel(io.BytesIO(data_upload[data_name]), sheet_name='Headlines')

print('Upload the completed, two-rater gold-standard workbook')
gold_upload = files.upload()
if not gold_upload:
    raise ValueError('No file uploaded. Please upload your completed gold annotation workbook.')
gold_name = next(iter(gold_upload))
gold = pd.read_excel(io.BytesIO(gold_upload[gold_name]), sheet_name='Annotate')

print(f'Dataset: {data_name} — {len(dataset):,} rows')
print(f'Gold annotations: {gold_name} — {len(gold):,} rows')

In [ ]:
# Check required columns and make IDs consistent text values.
DATA_REQUIRED = {'record_id', 'headline'}
GOLD_REQUIRED = {
    'record_id', 'headline',
    'rater_1_sentiment', 'rater_2_sentiment', 'consensus_sentiment',
    'rater_1_topic', 'rater_2_topic', 'consensus_topic'
}
missing_data = DATA_REQUIRED - set(dataset.columns)
missing_gold = GOLD_REQUIRED - set(gold.columns)
if missing_data:
    raise ValueError(f'Dataset is missing columns: {sorted(missing_data)}')
if missing_gold:
    raise ValueError(f'Gold workbook is missing columns: {sorted(missing_gold)}')

for frame in (dataset, gold):
    frame['record_id'] = frame['record_id'].astype(str).str.strip()
    frame['headline'] = frame['headline'].fillna('').astype(str).str.strip()
if dataset['record_id'].duplicated().any():
    raise ValueError('record_id must be unique in the analysis dataset.')
if gold['record_id'].duplicated().any():
    raise ValueError('record_id must be unique in the gold workbook.')
if dataset['headline'].eq('').any() or gold['headline'].eq('').any():
    raise ValueError('A headline is blank. Check the uploaded workbooks.')

print('Dataset preview')
display(dataset.head(5))
print('Gold annotation preview')
display(gold[['record_id','headline','rater_1_sentiment','rater_2_sentiment','consensus_sentiment','rater_1_topic','rater_2_topic','consensus_topic']].head(5))

## 2. Review or change the category lists

The starter lists below match the dropdowns in the supplied gold template. If your class wants different labels, change these lists **before annotation**, and update the gold workbook’s Codebook and annotation options so every annotator uses the same labels. The notebook will check LLM outputs against the lists you set here.

In [ ]:
sentiment_categories = ['مثبت', 'منفی', 'خنثی']
topic_categories = [
    'سیاست و حکومت', 'اقتصاد', 'جامعه و حقوق', 'بین‌الملل و درگیری',
    'فرهنگ و دین', 'سلامت و آموزش', 'ورزش', 'علم و فناوری', 'سایر / نامشخص'
]

print('Sentiment labels')
display(pd.DataFrame({'label': sentiment_categories}))
print('Topic labels')
display(pd.DataFrame({'label': topic_categories}))

## 3. Write your prompt in Persian

Edit `PROMPT_FA` in the next cell. Explain how to make both decisions, especially what counts as neutral sentiment and how to choose one main topic. You can improve the wording after the pilot, but do not copy the 100 gold headlines or their labels into the prompt.

In [ ]:
PROMPT_FA = """
شما در حال دسته‌بندی تیترهای خبری فارسی هستید. فقط بر اساس متن همان تیتر تصمیم بگیرید.

برای احساس، مثبت یعنی رویداد یا پیامد به شکلی مطلوب بازنمایی شده است؛ منفی یعنی نامطلوب بازنمایی شده است؛ خنثی یعنی تیتر عمدتاً خبری است یا ارزیابی روشن مثبت/منفی ندارد.

برای موضوع، فقط مهم‌ترین موضوع تیتر را انتخاب کنید. اگر هیچ موضوعی مناسب نیست یا اطلاعات کافی نیست، «سایر / نامشخص» را انتخاب کنید.

هر تیتر را مستقل بررسی کنید. از دانش بیرونی درباره ناشر یا رویداد استفاده نکنید. برای هر برچسب یک شاهد کوتاه از خود تیتر بنویسید.
"""

MODEL = 'gemini-3.8-flash'  # You may replace this with a model available to your account.
BATCH_SIZE = 10             # Smaller batches are easier to inspect and recover.

def build_request_prompt(records):
    return f"""{PROMPT_FA.strip()}

برچسب‌های مجاز احساس: {json.dumps(sentiment_categories, ensure_ascii=False)}
برچسب‌های مجاز موضوع: {json.dumps(topic_categories, ensure_ascii=False)}

برای هر ورودی دقیقاً یک برچسب احساس و یک برچسب موضوع برگردانید. شناسه را عیناً حفظ کنید.
ورودی‌ها:
{json.dumps(records, ensure_ascii=False)}"""

class OneAnnotation(BaseModel):
    record_id: str = Field(description='The input record ID, copied exactly')
    sentiment: str = Field(description='Exactly one allowed sentiment label')
    topic: str = Field(description='Exactly one allowed topic label')
    evidence: str = Field(description='A short phrase from the headline supporting the labels')

class AnnotationBatch(BaseModel):
    predictions: list[OneAnnotation]

print('Prompt is ready. Read it aloud, edit it in Persian, then run the connection cell.')

## 4. Connect to Gemini securely

Create or obtain a Gemini API key through Google AI Studio. In Colab, open the **key icon (Secrets)** in the left sidebar, add a secret named `GEMINI_API_KEY`, paste the key there, and enable notebook access. Do not paste the key into a code cell or share it in your notebook.

In [ ]:
api_key = userdata.get('GEMINI_API_KEY')
if not api_key:
    raise ValueError('Add GEMINI_API_KEY in Colab Secrets and allow this notebook to access it.')
client = genai.Client(api_key=api_key)
print('Gemini client connected. The key was not displayed.')

## 5. Pilot on the gold sample

The pilot sends the 100 gold headlines to the model. Check the prompt and codebook first. Each batch is one API request. The code retries a failed batch up to three times and keeps the returned record IDs so predictions can be matched to the right headline.

In [ ]:
def classify_frame(frame, batch_size=BATCH_SIZE, pause_seconds=0.3):
    rows = frame[['record_id', 'headline']].copy()
    rows['record_id'] = rows['record_id'].astype(str)
    all_predictions = []
    total_batches = math.ceil(len(rows) / batch_size)
    for batch_no, start in enumerate(range(0, len(rows), batch_size), start=1):
        batch = rows.iloc[start:start+batch_size]
        records = batch.to_dict(orient='records')
        prompt = build_request_prompt(records)
        last_error = None
        for attempt in range(1, 4):
            try:
                response = client.models.generate_content(
                    model=MODEL,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=0,
                        response_mime_type='application/json',
                        response_schema=AnnotationBatch,
                    ),
                )
                parsed = AnnotationBatch.model_validate_json(response.text)
                predictions = [p.model_dump() for p in parsed.predictions]
                expected = set(batch['record_id'])
                returned = [str(p['record_id']) for p in predictions]
                if set(returned) != expected or len(returned) != len(expected):
                    raise ValueError(f'Batch IDs do not match. Expected {len(expected)}, got {len(returned)}.')
                all_predictions.extend(predictions)
                last_error = None
                break
            except Exception as exc:
                last_error = exc
                if attempt < 3:
                    time.sleep(2 ** attempt)
        if last_error is not None:
            raise RuntimeError(f'Batch {batch_no}/{total_batches} failed after 3 attempts: {last_error}')
        if batch_no % 2 == 0 or batch_no == total_batches:
            print(f'Completed {batch_no}/{total_batches} batches')
        time.sleep(pause_seconds)
    result = pd.DataFrame(all_predictions)
    result['record_id'] = result['record_id'].astype(str)
    bad_sentiment = sorted(set(result['sentiment']) - set(sentiment_categories))
    bad_topic = sorted(set(result['topic']) - set(topic_categories))
    if bad_sentiment or bad_topic:
        raise ValueError(f'Unexpected labels. Sentiment={bad_sentiment}; topic={bad_topic}. Update prompt/categories and retry.')
    return result

gold_scoring = gold.dropna(subset=['consensus_sentiment','consensus_topic']).copy()
gold_scoring = gold_scoring[
    gold_scoring['consensus_sentiment'].astype(str).str.strip().ne('') &
    gold_scoring['consensus_topic'].astype(str).str.strip().ne('')
]
if len(gold_scoring) < 20:
    raise ValueError(f'Only {len(gold_scoring)} complete consensus rows found. Complete the consensus columns first (aim for all 100).')

gold_predictions = classify_frame(gold_scoring)
gold_results = gold_scoring.merge(gold_predictions, on='record_id', how='inner', validate='one_to_one')
display(gold_results[['record_id','headline','consensus_sentiment','sentiment','consensus_topic','topic','evidence']].head(12))

## 6. Check human agreement and LLM agreement with the gold labels

**Human–human reliability:** compare Rater 1 and Rater 2 on the 100-row file. **LLM–gold agreement:** compare the model with the consensus labels. Cohen’s kappa adjusts observed agreement for agreement expected by chance. The result is descriptive: a small gold sample and ambiguous headlines can make the value unstable.

In [ ]:
def agreement_summary(frame, left, right, label_name):
    pair = frame[[left, right]].copy()
    for col in (left, right):
        pair[col] = pair[col].fillna('').astype(str).str.strip()
    pair = pair[(pair[left] != '') & (pair[right] != '')]
    if len(pair) == 0:
        return {'task': label_name, 'pairs': 0, 'exact_agreement': np.nan, 'cohen_kappa': np.nan}
    agreement = (pair[left] == pair[right]).mean()
    try:
        kappa = cohen_kappa_score(pair[left], pair[right])
    except Exception:
        kappa = np.nan
    return {'task':label_name, 'pairs':len(pair), 'exact_agreement':agreement, 'cohen_kappa':kappa}

human_irr = pd.DataFrame([
    agreement_summary(gold, 'rater_1_sentiment', 'rater_2_sentiment', 'Human raters — sentiment'),
    agreement_summary(gold, 'rater_1_topic', 'rater_2_topic', 'Human raters — topic'),
])
llm_irr = pd.DataFrame([
    agreement_summary(gold_results, 'consensus_sentiment', 'sentiment', 'LLM vs consensus — sentiment'),
    agreement_summary(gold_results, 'consensus_topic', 'topic', 'LLM vs consensus — topic'),
])
agreement_table = pd.concat([human_irr, llm_irr], ignore_index=True)
agreement_table['exact_agreement'] = agreement_table['exact_agreement'].map(lambda x: f'{x:.1%}' if pd.notna(x) else 'not available')
agreement_table['cohen_kappa'] = agreement_table['cohen_kappa'].map(lambda x: f'{x:.3f}' if pd.notna(x) else 'not defined')
display(agreement_table)

print('Gold label counts')
display(gold_scoring['consensus_sentiment'].value_counts().rename_axis('sentiment').to_frame('n'))
display(gold_scoring['consensus_topic'].value_counts().rename_axis('topic').to_frame('n'))

In [ ]:
# Compare consensus labels with model labels using confusion matrices.
def rtl(text):
    return get_display(arabic_reshaper.reshape(str(text)))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, task, labels, gold_col, pred_col in [
    (axes[0], 'Sentiment: consensus vs LLM', sentiment_categories, 'consensus_sentiment', 'sentiment'),
    (axes[1], 'Topic: consensus vs LLM', topic_categories, 'consensus_topic', 'topic'),
]:
    cm = confusion_matrix(gold_results[gold_col], gold_results[pred_col], labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[rtl(x) for x in labels], yticklabels=[rtl(x) for x in labels], ax=ax)
    ax.set_title(task)
    ax.set_xlabel('LLM label')
    ax.set_ylabel('Human consensus')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

## 7. Run the full 2,000-headline annotation

Review the pilot results before continuing. Full annotation makes additional API requests. Set the confirmation variable to `True` only when you are ready. If you changed a prompt or categories after inspecting the pilot, rerun the pilot first.

In [ ]:
RUN_FULL_DATASET = False  # Change to True, then run this cell to classify every uploaded headline.

if not RUN_FULL_DATASET:
    print('Full run skipped. Review the pilot, then change RUN_FULL_DATASET to True and rerun this cell.')
else:
    full_predictions = classify_frame(dataset)
    annotated = dataset.merge(full_predictions, on='record_id', how='left', validate='one_to_one')
    print(f'Annotated {annotated.sentiment.notna().sum():,} of {len(annotated):,} headlines.')
    display(annotated.head(10))

## 8. Explore the full-sample results

These figures summarize the model annotations for this random 2,000-headline sample. The year plot shows proportions within each year, which is easier to compare than raw counts when sample sizes differ.

In [ ]:
if 'annotated' not in globals():
    raise ValueError('Run the full dataset annotation cell first.')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sent_order = [x for x in sentiment_categories if x in set(annotated['sentiment'])]
topic_order = [x for x in topic_categories if x in set(annotated['topic'])]
sns.countplot(data=annotated, x='sentiment', order=sent_order, ax=axes[0])
axes[0].set_title('Predicted sentiment counts')
axes[0].set_xlabel('Sentiment')
axes[0].set_xticklabels([rtl(x) for x in sent_order], rotation=25, ha='right')
sns.countplot(data=annotated, y='topic', order=topic_order, ax=axes[1])
axes[1].set_title('Predicted topic counts')
axes[1].set_ylabel('Topic')
axes[1].set_yticklabels([rtl(x) for x in topic_order])
plt.tight_layout()
plt.show()

if 'solar_year' in annotated.columns:
    by_year = pd.crosstab(annotated['solar_year'], annotated['sentiment'], normalize='index').reindex(columns=sentiment_categories, fill_value=0)
    by_year.plot(kind='line', marker='o', figsize=(11,5))
    plt.title('Share of predicted sentiment by Solar Hijri year')
    plt.xlabel('Solar Hijri year')
    plt.ylabel('Share of headlines')
    plt.legend([rtl(x) for x in by_year.columns], title='Sentiment')
    plt.tight_layout()
    plt.show()

topic_sentiment = pd.crosstab(annotated['topic'], annotated['sentiment'], normalize='index').reindex(index=topic_order, columns=sentiment_categories, fill_value=0)
plt.figure(figsize=(9, max(4, len(topic_order)*0.55)))
sns.heatmap(topic_sentiment, annot=True, fmt='.0%', cmap='YlGnBu', xticklabels=[rtl(x) for x in topic_sentiment.columns], yticklabels=[rtl(x) for x in topic_sentiment.index])
plt.title('Sentiment mix within each predicted topic')
plt.xlabel('Predicted sentiment')
plt.ylabel('Predicted topic')
plt.tight_layout()
plt.show()

## 9. Save and download your results

The output workbook contains the row ID, headline, outlet/year context, both model labels, and a short text-based evidence field. Keep the model, prompt, and category definitions with your report so someone else can understand how the labels were produced.

In [ ]:
if 'annotated' not in globals():
    raise ValueError('Run the full dataset annotation cell first.')

output_file = 'persian_headlines_annotated_2000.xlsx'
metadata = pd.DataFrame({
    'item':['model','prompt_language','sentiment_categories','topic_categories','analysis_rows'],
    'value':[MODEL,'Persian', ' | '.join(sentiment_categories), ' | '.join(topic_categories), len(annotated)]
})
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    annotated.to_excel(writer, index=False, sheet_name='Annotations')
    metadata.to_excel(writer, index=False, sheet_name='Run info')
files.download(output_file)
print(f'Saved {output_file}')

## Interpretation and limitations

- Cohen’s kappa is an agreement statistic, not proof that the codebook is correct. Report the number of complete rater pairs and the observed agreement with kappa.
- Use the consensus column only after annotators resolve disagreements. Keep the original rater columns so reliability remains auditable.
- This is a random sample from a larger archive collection, not a census of all Persian news or all Iranian media. The source collection uses sampled archive dates and outlets.
- Sentiment is subjective, especially for short headlines. Record unclear cases and revise definitions before annotation, not midway through a run.
- LLM labels are predictions. Keep a human-checked gold sample and report both human–human and LLM–consensus agreement.
- Do not describe changes over time as causal effects; source mix, archive coverage, and language conventions may also change.